# Phase 6 — Dual-Path Stage 1 + Stage 2 Evaluation

Evaluate the Phase 6 trained Dual-Path Stage 1 (Path A causal + Path B UNet + FusionGate)
combined with the Stage 2 EDM-Karras diffusion decoder (loaded from prior causal training)
on the validation split. Compute the full metric battery:

- Deterministic: RMSE_final, Pearson, F1@p95, F1@p99
- Probabilistic: CRPS_log1p, spread, spread/RMSE ratio
- Climate (via aligned_eval): CDD, Rx1day, R10day biases, PSD distance

Inputs
------
- `epoch_best_dualpath.pth` : Phase 6 output (encoder + RCN + reg_head + dual_path)
- `epoch_last.pth`          : prior causal training (provides Stage 2 diffusion weights)
- `sigma_data` = 0.193      : recalibrated by Phase 6

Output
------
- `phase6_evaluation/phase6_eval_results.json`


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab — Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Imports + Constantes Phase 6 Eval ===
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from pathlib import Path
from omegaconf import OmegaConf

from st_cdgm.models.dual_path_stage1 import DualPathPredictor, PathBCNN, FusionGate
from st_cdgm.training.stage1_paths import (
    predict_mu_hr_dualpath,
    batch_lr_grid_last,
    _as_batched_hr,
)

# --- Chemins Drive ---
DRIVE_ROOT     = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N      = DRIVE_ROOT / 'oracle_9node' / 'seed_42'

# Path A + dual_path + new mu_HR (sortie Phase 6)
CKPT_DUALPATH  = ORACLE_9N / 'epoch_best_dualpath.pth'

# Stage 2 (diffusion) : provient du training Path C+ Option C 9-node original.
# Dans le layout reel des notebooks 9-node, epoch_last.pth est directement dans
# oracle_9node/seed_42/ (pas dans un sous-dossier ckpt_causal/).
CKPT_STAGE2    = ORACLE_9N / 'epoch_last.pth'

# Recalibrated sigma_data from Phase 6 (was 0.18856 in Phase 5)
SIGMA_DATA_NEW = 0.193

# Eval params (match Phase 5 evaluation)
K_SAMPLES      = 4
N_STEPS        = 32
N_BATCHES_EVAL = 90  # match Phase 5 in-distribution eval

# Output dir
OUT_DIR        = ORACLE_9N / 'phase6_evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Validations ---
assert CKPT_DUALPATH.exists(), f'Phase 6 dual-path checkpoint introuvable : {CKPT_DUALPATH}'
assert CKPT_STAGE2.exists(),   f'Stage 2 checkpoint introuvable : {CKPT_STAGE2}'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cell 2] DEVICE         = {DEVICE}')
print(f'[Cell 2] CKPT_DUALPATH  = {CKPT_DUALPATH}')
print(f'[Cell 2] CKPT_STAGE2    = {CKPT_STAGE2}')
print(f'[Cell 2] SIGMA_DATA_NEW = {SIGMA_DATA_NEW}')
print(f'[Cell 2] K_SAMPLES      = {K_SAMPLES}  N_STEPS = {N_STEPS}  N_BATCHES_EVAL = {N_BATCHES_EVAL}')
print(f'[Cell 2] OUT_DIR        = {OUT_DIR}')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Forcer batch_size=1 pour la compatibilite single-sample (IterableDataset)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# --- Ajout metapaths 9-node ---
OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- Dates ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

# --- convert_sample_to_batch (identique training, avec lr_grid pour batch_lr_grid_last) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

# Constantes pour HR shape utilises plus tard pour DualPathPredictor
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Charger Stage 2 (diffusion decoder) ===
# Pattern build_stack du v5_eval, override projection_class_embeddings_input_dim
# pour matcher le training 9-node (num_vars * conditioning_dim).
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

print(f'[Cell 4] Chargement Stage 2 : {CKPT_STAGE2}')
ck_s2 = torch.load(CKPT_STAGE2, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] Cles disponibles : {sorted(ck_s2.keys())[:12]} ...')

# --- Determiner num_vars depuis le checkpoint dual-path Phase 6 ---
# (encoder a num_vars metapaths : 6 pour ancien, 9 pour Option C 9-node)
_ck_dp_peek = torch.load(CKPT_DUALPATH, map_location='cpu', weights_only=False)
_enc_sd_peek = _ck_dp_peek.get('encoder_state_dict', {})
_metapath_names = set()
for k in _enc_sd_peek:
    if k.startswith('metapath_convs.'):
        name = k[len('metapath_convs.'):].split('__')[0]
        _metapath_names.add(name)
num_vars = len(_metapath_names)
del _ck_dp_peek, _enc_sd_peek
print(f'[Cell 4] num_vars detecte = {num_vars}  (metapaths trouves)')

# --- Build UNET_KWARGS avec override projection_class_embeddings_input_dim ---
UNET_KWARGS = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KWARGS and isinstance(UNET_KWARGS[_k], list):
        UNET_KWARGS[_k] = tuple(UNET_KWARGS[_k])
# Critical: 9-node training a injecte cette key
UNET_KWARGS['projection_class_embeddings_input_dim'] = num_vars * int(CONFIG.diffusion.conditioning_dim)
print(f'[Cell 4] projection_class_embeddings_input_dim = {UNET_KWARGS["projection_class_embeddings_input_dim"]}')

# --- Build diffusion module ---
edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])

diffusion_decoder = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height),
    width=int(CONFIG.diffusion.width),
    unet_kwargs=UNET_KWARGS,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=bool(CONFIG.diffusion.get('use_gradient_checkpointing', False)),
    conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
    edm_config=edm_cfg,
    causal_concat=True,
).to(DEVICE)

# --- Strip prefixes torch.compile / DDP ---
def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out

_diff_sd = _strip_prefixes(ck_s2.get('diffusion_state_dict'))
if _diff_sd is None:
    raise RuntimeError(
        f'diffusion_state_dict absent de {CKPT_STAGE2}. '
        f'Le checkpoint Stage 2 doit etre le epoch_last.pth du training Path C+ '
        f'Option C 9-node.'
    )
_missing, _unexpected = diffusion_decoder.load_state_dict(_diff_sd, strict=False)
if _missing:
    print(f'[Cell 4] missing keys : {len(_missing)} (premieres : {_missing[:3]})')
if _unexpected:
    print(f'[Cell 4] unexpected keys : {len(_unexpected)} (premieres : {_unexpected[:3]})')

# --- Override sigma_data avec la valeur recalibree Phase 6 ---
_old_sigma = diffusion_decoder.edm_config.sigma_data
diffusion_decoder.edm_config.sigma_data = SIGMA_DATA_NEW
print(f'[Cell 4] sigma_data : {_old_sigma:.5f} -> {SIGMA_DATA_NEW} (override Phase 6)')

# --- Freeze + eval ---
for p in diffusion_decoder.parameters():
    p.requires_grad_(False)
diffusion_decoder.eval()

_n_diff = sum(p.numel() for p in diffusion_decoder.parameters())
print(f'[Cell 4] Stage 2 loaded, sigma_data={SIGMA_DATA_NEW}, diffusion params = {_n_diff:,}')


In [ ]:
# === Cell 5 : Charger Phase 6 Dual-Path Stage 1 ===
# Reproduit le pattern training Cell 4 + Cell 5 (build encoder, RCN, head,
# dual_path) en mode eval/inference seulement.
import re
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

def _parse_encoder_metapaths_from_ckpt(enc_sd):
    """Infer ordered (name, src, rel, target) list from checkpoint encoder keys.
    Format: metapath_convs.{name}__{src}__{rel}__{target}.{param}
    """
    seen = {}
    order = []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]
        src  = parts[1]
        rel  = parts[2]
        tgt  = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt)
            order.append(name)
    return [(n,) + seen[n] for n in order]

def _build_encoder_from_ckpt(enc_sd, CONFIG, device):
    """Build encoder whose metapath configs match exactly the checkpoint keys."""
    parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
    print(f'  Metapaths detectes : {[t[0] for t in parsed]}')
    cfgs = [
        IntelligibleVariableConfig(name=name, meta_path=(src, rel, tgt), pool='mean')
        for name, src, rel, tgt in parsed
    ]
    enc = IntelligibleVariableEncoder(
        configs=cfgs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(device)
    return enc, len(cfgs)

# --- Charger checkpoint ---
print(f'[Cell 5] Chargement : {CKPT_DUALPATH}')
ck_dp = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
print(f'[Cell 5] Cles dans checkpoint : {sorted(ck_dp.keys())[:12]} ...')

def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd

_enc_sd = _clean_sd(ck_dp.get('encoder_state_dict', {}))

# --- Build encoder (taille auto-detectee depuis ckpt) ---
torch.manual_seed(SEED); np.random.seed(SEED)
print('[Cell 5] Inference structure encoder depuis checkpoint...')
encoder, num_vars = _build_encoder_from_ckpt(_enc_sd, CONFIG, DEVICE)

# --- RCN driver dim via probe ---
_probe_b = next(iter(val_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
RCN_DRIVER_DIM = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=num_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=RCN_DRIVER_DIM,
    reconstruction_dim=RCN_DRIVER_DIM,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

# --- DualPath (architecture identique au training Phase 6) ---
PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
GATE_MAX_MEAN        = 0.40

dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

# --- Charger tous les state_dicts (strict=False pour path_b_bias compat) ---
def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] charge depuis "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] ERREUR load avec "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] AUCUNE cle valide dans {keys}')
    return False

_safe_load(encoder,         ck_dp, ['encoder_state_dict'],                            'encoder')
_safe_load(rcn_cell,        ck_dp, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
_safe_load(regression_head, ck_dp, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path,       ck_dp, ['dual_path_state_dict'],                          'dual_path')

# --- Freeze tout en eval ---
for m in (encoder, rcn_cell, regression_head, dual_path):
    for p in m.parameters():
        p.requires_grad_(False)
    m.eval()

# --- Verifier A_dag ---
_rcn_core = rcn_cell
if hasattr(_rcn_core, '_orig_mod'):
    _rcn_core = _rcn_core._orig_mod
if hasattr(_rcn_core, 'A_dag'):
    _A_dag = _rcn_core.A_dag.detach()
    print(f'[Cell 5] A_dag  shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
          f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')
else:
    print('[Cell 5] WARN : A_dag absent')

# --- Stats ---
_n_enc = sum(p.numel() for p in encoder.parameters())
_n_rcn = sum(p.numel() for p in rcn_cell.parameters())
_n_rh  = sum(p.numel() for p in regression_head.parameters())
_n_dp  = sum(p.numel() for p in dual_path.parameters())
print(f'[Cell 5] Param counts : enc={_n_enc:,}  rcn={_n_rcn:,}  head={_n_rh:,}  dual_path={_n_dp:,}')
print(f'[Cell 5] path_b_bias  : {float(dual_path.path_b_bias.item()):+.5f}')
print(f'[Cell 5] sigma_data Stage 2 effectif : {diffusion_decoder.edm_config.sigma_data}')


In [ ]:
# === Cell 6 : Inference loop apples-to-apples (match v5_eval Cell 10) ===
import time as _time
import numpy as _np

# Constants matching v5_evaluation Cell 10 (Phase 7 OOD protocol)
N_TIMES_EVAL    = 365      # full year (match v5_eval N_TIMES_OOD)
K_SAMPLES_EVAL  = 12       # match v5_eval K_SAMPLES_OOD (was 4 in old Phase 6)
N_STEPS_DIFF    = 18       # match v5_eval N_STEPS_DIFF (was 32)

print(f"[Cell 6] Apples-to-apples protocol :")
print(f"  N_TIMES_EVAL    = {N_TIMES_EVAL}")
print(f"  K_SAMPLES_EVAL  = {K_SAMPLES_EVAL}")
print(f"  N_STEPS_DIFF    = {N_STEPS_DIFF}")
print(f"  Total diffusion steps  = {N_TIMES_EVAL * K_SAMPLES_EVAL * N_STEPS_DIFF:,}")


@torch.no_grad()
def predict_dualpath_apples(batch, K=K_SAMPLES_EVAL, n_steps=N_STEPS_DIFF):
    """Same wiring as predict_dualpath (Path A + Path B + gate + K-sample Stage 2),
    but uses K=12 and n_steps=18 to match v5_eval Cell 10."""
    mu_A, mu_B, mu_total, gate = predict_mu_hr_dualpath(
        batch,
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        dual_path=dual_path,
        builder=builder, device=DEVICE,
    )
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    samples = []
    for _k in range(K):
        out = diffusion_decoder.sample(
            conditioning=None,
            num_steps=n_steps,
            scheduler_type='edm_karras',
            apply_constraints=False,
            mu_HR=mu_total,
            baseline_log=bl,
        )
        r = out.residual if hasattr(out, 'residual') else out
        hr_pred = bl + mu_total + r
        samples.append(hr_pred)
    ensemble = torch.stack(samples, dim=0)  # [K, 1, 1, H, W]
    return ensemble


# --- Collection loop matching v5_eval collect_predictions_for_gcm pattern ---
ens_log, truths_log, times_list = [], [], []

t0 = _time.time()
n_seen = 0
for bi, batch_list in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
    if n_seen >= N_TIMES_EVAL:
        break
    for micro in batch_list:
        if n_seen >= N_TIMES_EVAL:
            break

        # K-sample ensemble : [K, 1, 1, H, W]
        ens = predict_dualpath_apples(micro, K=K_SAMPLES_EVAL, n_steps=N_STEPS_DIFF)
        # Reduce to (K, H, W) — drop batch=1 and channel=1
        while ens.dim() > 3:
            ens = ens.squeeze(1)
        ens_arr = ens.cpu().numpy()                          # (K, H, W)

        # Truth in log1p : baseline + residual at last timestep
        target_res = micro['residual'][-1].to(DEVICE)
        bl_t       = micro['baseline'][-1].to(DEVICE)
        if bl_t.dim() == target_res.dim() - 1:
            bl_t = bl_t.unsqueeze(0)
        bl_t = torch.nan_to_num(bl_t, nan=0.0)
        truth = (bl_t + target_res).cpu()
        # Squeeze to (H, W)
        while truth.dim() > 2:
            truth = truth.squeeze(0)
        truth_arr = truth.numpy()                            # (H, W)

        ens_log.append(ens_arr)
        truths_log.append(truth_arr)

        # Time : prefer micro['time'] (sequence -> last), fallback synthetic later
        tv = micro.get('time')
        if tv is not None:
            try:
                if hasattr(tv, '__len__') and not isinstance(tv, str):
                    tv = tv[-1]
            except Exception:
                pass
            times_list.append(tv)
        else:
            times_list.append(None)

        n_seen += 1

    if (n_seen) % 20 == 0 and n_seen > 0:
        elapsed = _time.time() - t0
        rate = n_seen / max(elapsed, 1e-6)
        eta  = (N_TIMES_EVAL - n_seen) / max(rate, 1e-6)
        print(f'  [Cell 6] {n_seen:3d}/{N_TIMES_EVAL}  elapsed={elapsed:5.0f}s  '
              f'eta={eta:5.0f}s  ({rate*60:.2f} samples/min)')

# Stack into (K, T, H, W) and (T, H, W)
ens_log1p_KT  = _np.stack(ens_log,    axis=1)   # (K, T, H, W)
truth_log1p_T = _np.stack(truths_log, axis=0)   # (T, H, W)

print()
print(f'[Cell 6] Done in {_time.time() - t0:.0f}s ({(_time.time()-t0)/60:.1f} min)')
print(f'[Cell 6] ens_log1p_KT  shape = {ens_log1p_KT.shape}  (K, T, H, W)')
print(f'[Cell 6] truth_log1p_T shape = {truth_log1p_T.shape}  (T, H, W)')
print(f'[Cell 6] times collected      = {len(times_list)}  '
      f'(non-null: {sum(1 for t in times_list if t is not None)})')


In [ ]:
# === Cell 7 : probabilistic_metrics (verbatim from v5_eval Cell 10) ===
import numpy as np


def _crps_empirical_fast(samples, obs):
    """CRPS empirique vectorise via tri (O(K log K) par point).

    samples : (K, ...) array, ensemble
    obs     : (...,) array, observation
    return  : (...,) array, CRPS par point
    Formule : E|X - y| - 0.5 * E|X - X'|, avec
              0.5 * (1/K^2) * sum_ij |xi - xj| = (1/K^2) * sum_k (2k - K - 1) * x_(k)
    """
    K = samples.shape[0]
    term1 = np.nanmean(np.abs(samples - obs[None]), axis=0)
    s = np.sort(samples, axis=0)
    k_idx = np.arange(1, K + 1).reshape((K,) + (1,) * (s.ndim - 1)).astype(np.float64)
    weights = 2.0 * k_idx - K - 1.0
    term2 = np.sum(weights * s, axis=0) / (K * K)
    return term1 - term2


def _crps_clim_per_pixel(truth):
    """CRPS de la climato empirique (distribution par pixel sur l'axe temps).

    Pour X, X' iid ~ distribution-truth(h,w) et y ~ idem :
        CRPS_clim(h,w) = E|X - y| - 0.5 * E|X - X'| = 0.5 * E|X - X'|
    (car E|X-Y| = E|X-X'| pour des copies iid).
    """
    T = truth.shape[0]
    t_sorted = np.sort(truth, axis=0)
    k_idx = np.arange(1, T + 1).reshape((T, 1, 1)).astype(np.float64)
    weights = 2.0 * k_idx - T - 1.0
    return np.sum(weights * t_sorted, axis=0) / (T * T)


def _rank_histogram(samples, truth):
    """Histogramme de Talagrand : rang de truth parmi les K samples (K+1 bins)."""
    K = samples.shape[0]
    rank = (samples < truth[None]).sum(axis=0).astype(np.int64)
    hist, _ = np.histogram(rank.flatten(), bins=np.arange(K + 2) - 0.5)
    return hist.astype(int).tolist()


def probabilistic_metrics(ens_log1p, truth_log1p):
    """Calcule toutes les metriques probabilistes apres conversion log1p -> mm/jour.

    ens_log1p   : (K, T, H, W) ensemble en espace log1p
    truth_log1p : (T, H, W) verite en espace log1p
    """
    ens   = np.expm1(np.clip(ens_log1p.astype(np.float64),   0.0, None))
    truth = np.expm1(np.clip(truth_log1p.astype(np.float64), 0.0, None))

    pred_mean = ens.mean(axis=0)                         # (T, H, W)
    err2 = (pred_mean - truth) ** 2
    rmse_global = float(np.sqrt(np.nanmean(err2)))
    rmse_map_t = np.sqrt(np.nanmean(err2, axis=0))       # (H, W)

    ens_var = ens.var(axis=0)                            # (T, H, W)
    spread_global = float(np.sqrt(np.nanmean(ens_var)))
    spread_skill_ratio = float(spread_global / max(rmse_global, 1e-9))

    crps_model = _crps_empirical_fast(ens, truth)        # (T, H, W)
    crps_model_global = float(np.nanmean(crps_model))

    crps_clim_map = _crps_clim_per_pixel(truth)          # (H, W)
    crps_clim_global = float(np.nanmean(crps_clim_map))

    crps_ss = 1.0 - crps_model_global / max(crps_clim_global, 1e-9)

    hist = _rank_histogram(ens, truth)
    K = int(ens.shape[0])
    expected_per_bin = float(truth.size / (K + 1))
    chi2_uniform = float(sum((c - expected_per_bin) ** 2 / expected_per_bin for c in hist))

    return {
        "K_samples": K,
        "n_times": int(ens.shape[1]),
        "grid": [int(truth.shape[-2]), int(truth.shape[-1])],
        "rmse_global_mm": rmse_global,
        "rmse_map_mean_mm": float(np.nanmean(rmse_map_t)),
        "rmse_map_max_mm": float(np.nanmax(rmse_map_t)),
        "spread_global_mm": spread_global,
        "spread_skill_ratio": spread_skill_ratio,
        "crps_model_global_mm": crps_model_global,
        "crps_clim_global_mm": crps_clim_global,
        "crps_skill_score": float(crps_ss),
        "rank_histogram": hist,
        "rank_histogram_bins": list(range(len(hist))),
        "rank_histogram_chi2_vs_uniform": chi2_uniform,
        "_caveat": (
            "CRPS_clim computed from test-truth empirical distribution per pixel "
            "(includes the day under evaluation; slight optimistic bias for T~365). "
            "spread_skill_ratio ~1 = well-calibrated, <1 = under-dispersive, >1 = over-dispersive."
        ),
    }


prob_data = probabilistic_metrics(ens_log1p_KT, truth_log1p_T)

print('=' * 60)
print('PROBABILISTIC METRICS  (mm/day, identical to v5_eval Cell 10)')
print('=' * 60)
for k, v in prob_data.items():
    if isinstance(v, float):
        print(f'  {k:32s} = {v:.5f}')
    elif isinstance(v, (int,)):
        print(f'  {k:32s} = {v}')
    elif isinstance(v, list) and len(v) <= 20:
        print(f'  {k:32s} = {v}')
    elif isinstance(v, str):
        pass  # _caveat, too long
    else:
        print(f'  {k:32s} = <{type(v).__name__} len={len(v) if hasattr(v, "__len__") else "?"}>')


In [ ]:
# === Cell 8 : run_aligned_eval (indices + PSD, same call as v5_eval Cell 10) ===
import numpy as np
from st_cdgm.evaluation.aligned_eval import run_aligned_eval

# Times for aligned_eval : prefer collected timestamps, fallback to synthetic daily range
def _coerce_times_to_dt64(tlist, n):
    valid = [t for t in tlist if t is not None]
    if len(valid) == n:
        try:
            arr = np.array([np.datetime64(t) for t in valid], dtype='datetime64[D]')
            return arr
        except Exception as _e:
            print(f'[Cell 8] datetime64 coercion failed ({_e}); using synthetic range')
    print(f'[Cell 8] times incomplete ({len(valid)}/{n}); fallback synthetic daily range 2010-01-01+')
    return np.array([np.datetime64('2010-01-01') + np.timedelta64(i, 'D')
                     for i in range(n)], dtype='datetime64[D]')


times_for_aligned = _coerce_times_to_dt64(times_list, truth_log1p_T.shape[0])

# pred_mean (T, H, W) in log1p — required by run_aligned_eval(space='log1p')
pred_mean_log1p = ens_log1p_KT.mean(axis=0)  # (T, H, W)

aligned_out_path = OUT_DIR / 'phase6_aligned_metrics_apples.json'

aligned_result = run_aligned_eval(
    pred_fields=pred_mean_log1p,
    truth_fields=truth_log1p_T,
    times=times_for_aligned,
    out_path=str(aligned_out_path),
    gcm='ACCESS-CM2',
    run_variant='phase6_dualpath_apples',
    in_distribution=True,
    space='log1p',
    thresh=1.0,
    k_samples=K_SAMPLES_EVAL,
    psd_nx=int(H_HR),
    psd_ny=int(W_HR),
)

print('=' * 60)
print('ALIGNED METRICS  (run_aligned_eval, identical to v5_eval Cell 10)')
print('=' * 60)
for k, v in aligned_result.items():
    if isinstance(v, dict):
        print(f'  {k} :')
        for kk, vv in v.items():
            if isinstance(vv, float):
                print(f'    {kk:28s} = {vv:.5f}')
            else:
                print(f'    {kk:28s} = {vv}')
    elif isinstance(v, float):
        print(f'  {k:32s} = {v:.5f}')
    else:
        print(f'  {k:32s} = {v}')

print()
print(f'[Cell 8] Aligned JSON saved : {aligned_out_path}')


In [ ]:
# === Cell 9 : Save JSON + comparison vs Phase 5 Oracle reference ===
import json
from pathlib import Path

# --- Locate Phase 5 reference (aligned_metrics_ACCESS-CM2_causal.json) ---
_ref_data = None
_ref_path_used = None
for _candidate in [
    Path('/content/climate_data/results/6node-seed42/aligned_metrics_ACCESS-CM2_causal.json'),
    Path('/content/drive/MyDrive/climate_data/results/6node-seed42/aligned_metrics_ACCESS-CM2_causal.json'),
    Path('/content/drive/MyDrive/climate_data/results/aligned_metrics_ACCESS-CM2_causal.json'),
    REPO_DIR / 'results' / '6node-seed42' / 'aligned_metrics_ACCESS-CM2_causal.json',
    ORACLE_9N.parent.parent / 'results' / '6node-seed42' / 'aligned_metrics_ACCESS-CM2_causal.json',
]:
    try:
        if _candidate.exists():
            _ref_data = json.loads(_candidate.read_text(encoding='utf-8'))
            _ref_path_used = _candidate
            print(f'[Cell 9] Reference loaded : {_candidate}')
            break
    except Exception as _e:
        print(f'[Cell 9] {_candidate} : {type(_e).__name__}: {_e}')

if _ref_data is None:
    print('[Cell 9] No Phase 5 reference JSON found; comparison table will be partial.')

# --- Consolidate and persist ---
all_results = {
    'phase':              'phase6_dualpath_apples_to_apples',
    'protocol':           'cgan_rampal_vendored',
    'ckpt_dualpath':      str(CKPT_DUALPATH),
    'ckpt_stage2':        str(CKPT_STAGE2),
    'sigma_data':         float(SIGMA_DATA_NEW),
    'gcm':                'ACCESS-CM2',
    'in_distribution':    True,
    'n_times':            int(N_TIMES_EVAL),
    'k_samples':          int(K_SAMPLES_EVAL),
    'n_steps_diff':       int(N_STEPS_DIFF),
    'reference_path':     str(_ref_path_used) if _ref_path_used else None,
    'probabilistic_metrics_mm': prob_data,
    'aligned_metrics':    aligned_result,
}

out_path = OUT_DIR / 'phase6_eval_apples_to_apples.json'
out_path.write_text(json.dumps(all_results, indent=2, default=str), encoding='utf-8')
print(f'[Cell 9] Consolidated JSON saved : {out_path}')

# --- Comparison table ---
print()
print('=' * 80)
print('COMPARISON  Phase 6 DualPath (apples-to-apples)  vs  Phase 5 Oracle reference')
print('=' * 80)

_p6_indices = aligned_result.get('indices', {}) or {}
_p6_psd     = aligned_result.get('psd_distance', float('nan'))
_p6_rmse    = prob_data.get('rmse_global_mm', float('nan'))
_p6_crps    = prob_data.get('crps_model_global_mm', float('nan'))
_p6_crps_ss = prob_data.get('crps_skill_score', float('nan'))
_p6_sr      = prob_data.get('spread_skill_ratio', float('nan'))

if _ref_data is not None:
    _ref_indices = _ref_data.get('indices', {}) or {}
    _ref_psd     = _ref_data.get('psd_distance', float('nan'))
else:
    _ref_indices = {}
    _ref_psd     = float('nan')


def _gv(d, *keys, default=float('nan')):
    for k in keys:
        if k in d:
            try:
                return float(d[k])
            except Exception:
                return default
    return default


def _fmt(x):
    try:
        return f'{float(x):>14.5f}'
    except Exception:
        return f'{str(x):>14s}'


print(f'  {"Metric":<28} {"Phase 6":>14} {"Phase 5 ref":>14}   Comment')
print('  ' + '-' * 76)

# Probabilistic (no Phase 5 ref for these in aligned JSON, so '-')
print(f'  {"RMSE_global (mm)":<28} {_fmt(_p6_rmse)} {"-":>14}   from probabilistic_metrics')
print(f'  {"CRPS_global (mm)":<28} {_fmt(_p6_crps)} {"-":>14}')
print(f'  {"CRPS_skill_score":<28} {_fmt(_p6_crps_ss)} {"-":>14}')
print(f'  {"spread_skill_ratio":<28} {_fmt(_p6_sr)} {"-":>14}')

# Aligned indices
for label, *keys in [
    ('CDD_bias (days)',     'cdd_bias', 'CDD_bias', 'cdd'),
    ('Rx1day_bias (mm)',    'rx1day_bias', 'Rx1day_bias', 'rx1day'),
    ('R10day_bias (days)',  'r10day_bias', 'R10day_bias', 'r10day'),
    ('R95p_bias',           'r95p_bias',  'R95p_bias',  'r95p'),
    ('SDII_bias',           'sdii_bias',  'SDII_bias',  'sdii'),
]:
    p6_v  = _gv(_p6_indices,  *keys)
    ref_v = _gv(_ref_indices, *keys)
    print(f'  {label:<28} {_fmt(p6_v)} {_fmt(ref_v)}')

print(f'  {"PSD_distance":<28} {_fmt(_p6_psd)} {_fmt(_ref_psd)}')
print('  ' + '-' * 76)

# --- Verdict ---
print()
print('=' * 80)
print('VERDICT  (Phase 6 DualPath vs Phase 5 Oracle reference, ACCESS-CM2 ID)')
print('=' * 80)
if _ref_data is not None:
    _d_psd = _p6_psd - _ref_psd
    print(f'  Delta PSD_distance     : {_d_psd:+.5f}   '
          f'({"BETTER" if _d_psd < 0 else "WORSE"})')
    for label, *keys in [
        ('CDD_bias',    'cdd_bias', 'CDD_bias', 'cdd'),
        ('Rx1day_bias', 'rx1day_bias', 'Rx1day_bias', 'rx1day'),
        ('R10day_bias', 'r10day_bias', 'R10day_bias', 'r10day'),
    ]:
        p6_v  = _gv(_p6_indices,  *keys)
        ref_v = _gv(_ref_indices, *keys)
        if not (p6_v != p6_v or ref_v != ref_v):  # both finite (NaN check)
            d = abs(p6_v) - abs(ref_v)
            tag = 'BETTER' if d < 0 else 'WORSE'
            print(f'  Delta |{label}|  : {d:+.5f}   ({tag})')
else:
    print('  [INFO] No reference loaded, verdict skipped.')

print()
print(f'[Cell 9] DONE.  Apples-to-apples JSON => {out_path}')
